In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity

# Koleksi lirik lagu

daftar_lirik = [

    "cinta ini membara...",

    "patah hati lagi...",

# tambahkan 3 lagu lagi

]

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(daftar_lirik)



query = ["cinta patah hati"]

skor = cosine_similarity( vectorizer.transform(query), X )



print(skor)

[[0.33333333 0.66666667]]


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

daftar_lirik = [
    "cinta ini membara dan tak pernah padam",  
    "patah hati lagi karena cinta yang pergi", 
    "membara rasa cinta di dalam dada",  
    "hati ini patah dan terluka lagi",  
    "malam dingin tanpa ada rasa cinta",  
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(daftar_lirik)


def cari_lagu(query_text):
    query_vec = vectorizer.transform([query_text])
    skor = cosine_similarity(query_vec, X)[0]
    hasil_peringkat = np.argsort(skor)[::-1].tolist()
    return hasil_peringkat

def precision_at_k(hasil, relevan, k=5):
    return sum(1 for d in hasil[:k] if d in relevan) / k

def recall(hasil, relevan):
    if len(relevan) == 0:
        return 0.0
    cocok = sum(1 for d in hasil if d in relevan)
    return cocok / len(relevan)


def f1(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def average_precision(hasil, relevan):
    if len(relevan) == 0:
        return 0.0
    hit, total = 0, 0.0
    for i, d in enumerate(hasil, start=1):
        if d in relevan:
            hit += 1
            total += hit / i
    return total / len(relevan)

ujians = [
    {"kueri": "cinta membara", "ground_truth": {0, 2}},
    {"kueri": "patah hati", "ground_truth": {1, 3}},
    {"kueri": "malam dingin", "ground_truth": {4}},
]

daftar_ap = []

print("=== HASIL EVALUASI MESIN PENCARI ===")
for test in ujians:
    q = test["kueri"]
    gt = test["ground_truth"]

    hasil_pencarian = cari_lagu(q)

    p = precision_at_k(hasil_pencarian, gt, k=5)
    r = recall(hasil_pencarian, gt)
    f = f1(p, r)
    ap = average_precision(hasil_pencarian, gt)
    daftar_ap.append(ap)

    print(f"\nKueri: '{q}'")
    print(f"Hasil Urutan (ID Lagu) : {hasil_pencarian}")
    print(f"Ground Truth (ID Relevan): {gt}")
    print(
        f"Precision@5: {p:.4f} | Recall: {r:.4f} | F1: {f:.4f} | AP: {ap:.4f}"
    )

map_score = sum(daftar_ap) / len(daftar_ap)
print(f"\n====================================")
print(f"MAP (Mean Average Precision): {map_score:.4f}")
print(f"====================================")

=== HASIL EVALUASI MESIN PENCARI ===

Kueri: 'cinta membara'
Hasil Urutan (ID Lagu) : [2, 0, 4, 1, 3]
Ground Truth (ID Relevan): {0, 2}
Precision@5: 0.4000 | Recall: 1.0000 | F1: 0.5714 | AP: 1.0000

Kueri: 'patah hati'
Hasil Urutan (ID Lagu) : [3, 1, 4, 2, 0]
Ground Truth (ID Relevan): {1, 3}
Precision@5: 0.4000 | Recall: 1.0000 | F1: 0.5714 | AP: 1.0000

Kueri: 'malam dingin'
Hasil Urutan (ID Lagu) : [4, 3, 2, 1, 0]
Ground Truth (ID Relevan): {4}
Precision@5: 0.2000 | Recall: 1.0000 | F1: 0.3333 | AP: 1.0000

MAP (Mean Average Precision): 1.0000
